This script generates the following rasters at 5000 m resolution:
1. Rainfall: annual rainfall, number of wet months in the year (averaged from 2000-2023)
2. Temperature: Seasonal mean temperature, seasonal diurnal temperature range over rabi, kharif and zaid (averaged from 2000-2023)
3. Mean elevation: from SRTM DEM

It also generates the following rasters at 30 m resolution, from SRTM DEM:
1. Elevation
2. Slope
3. Aspect

# Set up

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import ee
import geemap
ee.Authenticate()
ee.Initialize(project='ee-indiasat')

# Config

In [ ]:
err = ee.ErrorMargin(100)
year = 2023

india = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'India'))
kolar = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level2').filter(ee.Filter.eq("ADM2_NAME","Kolar"))
karnataka = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM1_NAME', "Karnataka")).first()

indiaDistricts = ee.FeatureCollection("projects/ee-indiasat/assets/india_district_boundaries")
indiaACZs = ee.FeatureCollection("projects/ee-mtpictd/assets/harsh/Agroclimatic_regions")
india = indiaACZs

In [ ]:
def getImgCollection(folder_path):
  asset_list = ee.data.listAssets({'parent': folder_path})['assets']
  image_list = [ee.Image(asset['name']) for asset in asset_list]
  return ee.ImageCollection(image_list)

def getFeatureCollection(folder_path):
  asset_list = ee.data.listAssets({'parent': folder_path})
  fc_ids = [a['id'] for a in asset_list['assets'] if a['type'] == 'TABLE']
  merged_fc = ee.FeatureCollection(fc_ids[0])
  for fc_id in fc_ids[1:]:
    merged_fc = merged_fc.merge(ee.FeatureCollection(fc_id))
  return merged_fc


In [ ]:
agroClimaticZones = [
    'Eastern Plateau & Hills Region',
    'Southern Plateau and Hills Region',
    'East Coast Plains & Hills Region',
    'Western Plateau and Hills Region',
    'Central Plateau & Hills Region',
    'Lower Gangetic Plain Region',
    'Middle Gangetic Plain Region',
    'Eastern Himalayan Region',
    'Western Himalayan Region',
    'Upper Gangetic Plain Region',
    'Trans Gangetic Plain Region',
    'West Coast Plains & Ghat Region',
    'Gujarat Plains & Hills Region',
    'Western Dry Region'
]

aczAcronymDict = {'Eastern Plateau & Hills Region': 'EPAHR',
                  'Southern Plateau and Hills Region': 'SPAHR',
                  'East Coast Plains & Hills Region': 'ECPHR',
                  'Western Plateau and Hills Region': 'WPAHR',
                  'Central Plateau & Hills Region': 'CPAHR',
                  'Lower Gangetic Plain Region': 'LGPR',
                  'Middle Gangetic Plain Region': 'MGPR',
                  'Eastern Himalayan Region': 'EHR',
                  'Western Himalayan Region': 'WHR',
                  'Upper Gangetic Plain Region': 'UGPR',
                  'Trans Gangetic Plain Region': 'TGPR',
                  'West Coast Plains & Ghat Region': 'WCPGR',
                  'Gujarat Plains & Hills Region': 'GPHR',
                  'Western Dry Region': 'WDR'
                }


# Export rainfall rasters

## Generate pan-India rainfall and wet months image

In [ ]:
# Averaged over the time period from start_date for year_count years
# Wet months are months with > 100 mm rainfall
def rainfall_layers(roi, start_date, year_count):
  chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterBounds(roi.geometry())
  start = ee.Date(start_date)
  years = ee.List.sequence(0, year_count - 1)

  def process_year(year_offset):
    year_start = start.advance(year_offset, 'year')
    year_end = year_start.advance(1, 'year')

    daily_rain = chirps.filterDate(year_start, year_end)
    total_annual = daily_rain.sum().rename('annual_rainfall')

    months = ee.List.sequence(0, 11)

    def monthly_sum(month_offset):
      month_start = year_start.advance(month_offset, 'month')
      month_end = month_start.advance(1, 'month')
      return chirps.filterDate(month_start, month_end).sum().rename('monthly_rainfall')

    monthly_sums = ee.ImageCollection(months.map(monthly_sum))
    wet_months = monthly_sums.map(lambda img: img.gt(100)).sum().rename('wet_months')

    return total_annual.addBands(wet_months).clip(roi.geometry())

  annual_stats = ee.ImageCollection(years.map(process_year))
  return annual_stats.mean().rename(['annual_rainfall', 'wet_months'])


Set up ROI

In [ ]:
# 'buffer' is the ROI
buffer = getFeatureCollection('projects/ee-mtpictd/assets/anoushka/india_buffers_200km').union().first()
india = indiaACZs.union().first()
diff = ee.Feature(buffer.geometry().difference(india.geometry()))

In [ ]:
rainfall_diff = rainfall_layers(diff, '2003-07-01', 21).clip(diff.geometry())
rainfall_diff.getInfo()

Output hidden; open in https://colab.research.google.com to view.

## Visualization

In [ ]:
rainfall_vis = {
    'min': 0,
    'max': 5000,
    'palette': ['green', 'yellow', 'red']
}

wet_months_vis = {
    'min': 0,
    'max': 12,
    'palette': ['lightblue', 'blue', 'darkblue', 'purple']
}


In [ ]:
Map = geemap.Map()
Map.addLayer(rainfall_diff.select('annual_rainfall'), rainfall_vis, "India Rainfall")
Map.addLayer(rainfall_diff.select('wet_months'), wet_months_vis, "India Wet Month")
Map.centerObject(india, 4)
Map

Map(center=[23.221038363300835, 79.46098884927167], controls=(WidgetControl(options=['position', 'transparent_…

## Export pan-India raster

In [ ]:
task = ee.batch.Export.image.toAsset(
  image=rainfall_diff,
  description=f'Diff Rainfall and Wet Months',
  assetId=f'projects/ee-mtpictd/assets/anoushka/diff2_annual_rainfall_wet_months_2000_2023',
  region=diff.geometry(),
  scale=5566,               # Native resolution of CHIRPS
  maxPixels=1e13

)

task.start()
print(f'Export task started.')


Export task started.


# Export temperature rasters

## Generate temperature images

In [ ]:
def mean_temp_images(roi, start_date, year_count):
  rabi = ee.List([1, 2, 11, 12])
  zaid = ee.List([3, 4, 5, 6])
  kharif = ee.List([7, 8, 9, 10])

  start = ee.Date(start_date)
  end = start.advance(year_count, 'year')

  daily_temp = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
                .select('temperature_2m')
                .filterDate(start, end)
                .filterBounds(roi.geometry()))

  def mean_temp(months):
      filtered_daily = daily_temp.filter(ee.Filter.inList('month', months))
      mean_daily_temp = filtered_daily.mean().subtract(273.15).clip(roi.geometry()).rename('mean_temp')
      return mean_daily_temp

  rabi_img = mean_temp(rabi).set('season', 'rabi')
  zaid_img = mean_temp(zaid).set('season', 'zaid')
  kharif_img = mean_temp(kharif).set('season', 'kharif')

  return ee.ImageCollection.fromImages([rabi_img, zaid_img, kharif_img])

In [ ]:
def diurnal_range_images(roi, start_date, year_count):
  rabi = ee.List([1, 2, 11, 12])
  zaid = ee.List([3, 4, 5, 6])
  kharif = ee.List([7, 8, 9, 10])

  start = ee.Date(start_date)
  end = start.advance(year_count, 'year')

  daily_temp = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
                .filterDate(start, end)
                .filterBounds(roi.geometry()))

  def diurnal_range(months):
    filtered_daily = daily_temp.filter(ee.Filter.inList('month', months))

    daily_range = filtered_daily.map(lambda img:
        img.select('temperature_2m_max').subtract(img.select('temperature_2m_min'))
          .rename('diurnal_range')
          .copyProperties(img, img.propertyNames())
    )
    return daily_range.mean().rename('mean_diurnal_range').clip(roi.geometry())

  rabi_img = diurnal_range(rabi).set('season', 'rabi')
  zaid_img = diurnal_range(zaid).set('season', 'zaid')
  kharif_img = diurnal_range(kharif).set('season', 'kharif')
  return ee.ImageCollection.fromImages([rabi_img, zaid_img, kharif_img])


In [ ]:
roi = indiaACZs.filter(ee.Filter.eq('regionname', 'Western Himalayan Region')).first()
start_date = '2003-07-01'
year_count = 21

mean_imgs = mean_temp_images(roi, start_date, year_count)
print(mean_imgs.getInfo())
diurnal_imgs = diurnal_range_images(roi, start_date, year_count)
print(diurnal_imgs.getInfo())

{'type': 'ImageCollection', 'bands': [], 'features': [{'type': 'Image', 'bands': [{'id': 'mean_temp', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'dimensions': [10, 10], 'origin': [72, 28], 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}], 'properties': {'system:footprint': {'type': 'Polygon', 'coordinates': [[[72.53003597156848, 35.92654385687588], [72.53122157619998, 35.92453555939204], [72.53305895942869, 35.920111644415044], [72.53372734996485, 35.91744213927926], [72.53770669798597, 35.91495112488069], [72.53827813703477, 35.91460944248706], [72.54132576079536, 35.91402601475425], [72.54242249951555, 35.91405092196422], [72.54353484836552, 35.91341538559438], [72.54411782774874, 35.91298035108519], [72.54908343435768, 35.912881511933236], [72.55048412341753, 35.912454810700424], [72.55227037690416, 35.90981543565704], [72.55368305277156, 35.908951381350946], [72.56166671163272, 35.90913842992183], [72.56196373172641, 35.90870893014314], [72.56888724527319, 

## Export images over ROI
Do this ACZ-wise, pan-India will hit computational time-out.

In [ ]:
# Export images over roi

for season in ['rabi', 'zaid', 'kharif']:
  img1 = ee.Image(mean_imgs.filter(ee.Filter.eq('season', season)).first()).rename(f'{season}_mean_temp')
  img2 = ee.Image(diurnal_imgs.filter(ee.Filter.eq('season', season)).first()).rename(f'{season}_diurnal_temp')
  img = img1.addBands(img2)
  task = ee.batch.Export.image.toAsset(
    image=img,
    description=f'WHR {season} Temp',
    assetId=f'projects/ee-ictd-dhruvi/assets/anoushka/india_temp_2000_2023/WHR_temp_{season}',
    region=roi.geometry(),
    scale=11132,
    maxPixels=1e13
  )
  task.start()
  print(f'Export task started for {season}.')


Export task started for rabi.
Export task started for zaid.
Export task started for kharif.


# Export mean elevation raster

No processing, just reproject (to 5000 m resolution) and export from SRTM-DEM.

In [ ]:
buffer = getFeatureCollection('projects/ee-mtpictd/assets/anoushka/india_buffers_200km').union().first()
india = indiaACZs.union().first()
diff = ee.Feature(buffer.geometry().difference(india.geometry()))

In [ ]:
roi = diff

elevation = ee.Image("USGS/SRTMGL1_003")
projection = elevation.projection()
mean_5km = (
    elevation
    .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=65535)
    .reproject(crs=projection, scale=5000)
).clip(roi.geometry())
elevation = elevation.clip(roi.geometry())

print(mean_5km.getInfo())

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
img = mean_5km
task = ee.batch.Export.image.toAsset(
  image=img,
  description=f'Buffer Mean Elevation',
  assetId=f'projects/ee-mtpictd/assets/anoushka/india_buffer_elevation_5km',
  region=roi.geometry(),
  scale=5000,
  maxPixels=1e13
)
task.start()
print(f'Export task started.')


Export task started.


# Export elevation, slope, aspect rasters

Get from ee.Terrain and SRTM DEM, then reproject and export

In [ ]:
roi = diff

elevation = ee.Image("USGS/SRTMGL1_003").clip(roi.geometry())
slope = ee.Terrain.slope(elevation)
aspect = ee.Terrain.aspect(elevation)

In [ ]:
for img, name in zip([elevation, slope, aspect], ['elevation', 'slope', 'aspect']):
  # if name in ['elevation', 'aspect']:
  #   continue
  task = ee.batch.Export.image.toAsset(
    image=img,
    description=f'India Buffer {name}',
    assetId=f'projects/ee-ictd-dhruvi/assets/anoushka/india_buffer_{name}_30m',
    region=roi.geometry(),
    scale=30,
    maxPixels=1e13
  )
  task.start()
  print(f'Export task started for {name}.')


Export task started for elevation.
Export task started for slope.
Export task started for aspect.


# Visualization

In [ ]:
inputLayers = ee.Image(0)

# -------------------------
# Total annual rainfall
# -------------------------
indiaRainfallWetMonths1 = ee.Image('projects/ee-mtpictd/assets/anoushka/india_annual_rainfall_wet_months_2000_2023')
indiaRainfallWetMonths2 = ee.Image('projects/ee-mtpictd/assets/anoushka/diff_annual_rainfall_wet_months_2000_2023')
indiaRainfallWetMonths3 = ee.Image('projects/ee-mtpictd/assets/anoushka/diff2_annual_rainfall_wet_months_2000_2023')

indiaRainfallWetMonths = (
    indiaRainfallWetMonths1
    .blend(indiaRainfallWetMonths2)
    .blend(indiaRainfallWetMonths3)
    .reproject(crs='EPSG:4326', scale=5000)
)

inputLayers = inputLayers.addBands(indiaRainfallWetMonths).select(['annual_rainfall', 'wet_months'])

# -------------------------
# Seasonal mean temperatures and diurnal ranges
# -------------------------

indiaMeanDiurnal = getImgCollection('projects/ee-ictd-dhruvi/assets/anoushka/india_temp_2000_2023')

rabiImage = (
    indiaMeanDiurnal.filter(ee.Filter.eq('season', 'rabi'))
    .mosaic()
    .reproject(crs='EPSG:4326', scale=5000)
)
kharifImage = (
    indiaMeanDiurnal.filter(ee.Filter.eq('season', 'kharif'))
    .mosaic()
    .reproject(crs='EPSG:4326', scale=5000)
)
zaidImage = (
    indiaMeanDiurnal.filter(ee.Filter.eq('season', 'zaid'))
    .mosaic()
    .reproject(crs='EPSG:4326', scale=5000)
)

inputLayers = inputLayers.addBands([rabiImage, kharifImage, zaidImage])

# -------------------------
# Mean elevation
# -------------------------
indiaMeanElevation = (
    ee.Image('projects/ee-mtpictd/assets/anoushka/india_elevation_5km')
    .blend(ee.Image('projects/ee-mtpictd/assets/anoushka/india_buffer_elevation_5km'))
    .reproject(crs='EPSG:4326', scale=5000)
    .rename('mean_elevation')
)

inputLayers = inputLayers.addBands(indiaMeanElevation)

print(inputLayers.getInfo())


{'type': 'Image', 'bands': [{'id': 'annual_rainfall', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'dimensions': [648, 313], 'origin': [1517, -740], 'crs': 'EPSG:4326', 'crs_transform': [0.04491576420597607, 0, 0, 0, -0.04491576420597607, 0]}, {'id': 'wet_months', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'dimensions': [648, 313], 'origin': [1517, -740], 'crs': 'EPSG:4326', 'crs_transform': [0.04491576420597607, 0, 0, 0, -0.04491576420597607, 0]}, {'id': 'rabi_mean_temp', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [0.04491576420597607, 0, 0, 0, -0.04491576420597607, 0]}, {'id': 'rabi_diurnal_temp', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [0.04491576420597607, 0, 0, 0, -0.04491576420597607, 0]}, {'id': 'kharif_mean_temp', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [0.04491576420597607, 0, 0, 0, -0

In [ ]:
inputLayers.bandNames().getInfo()

['annual_rainfall',
 'wet_months',
 'rabi_mean_temp',
 'rabi_diurnal_temp',
 'kharif_mean_temp',
 'kharif_diurnal_temp',
 'zaid_mean_temp',
 'zaid_diurnal_temp',
 'mean_elevation']

In [ ]:
rainfall_vis = {
    'min': 0,
    'max': 5000,
    'palette': ['green', 'yellow', 'red']
}

wet_months_vis = {
    'min': 0,
    'max': 12,
    'palette': ['lightblue', 'blue', 'darkblue', 'purple']
}

temp_vis = {
    'min': 0,
    'max': 50,
    'palette': ['lightblue', 'blue', 'darkblue', 'purple']
}

elevation_vis = {
    'min': 0,
    'max': 3000,
    'palette': [
        '0000ff',  # deep blue (sea level)
        '00ffff',  # cyan
        '00ff00',  # green (lowlands)
        'ffff00',  # yellow (mid elevation)
        'ff8000',  # orange
        'ff0000',  # red (high elevation)
        'ffffff'   # white (very high elevation)
    ]
}

In [ ]:
Map = geemap.Map()
Map.addLayer(inputLayers.select('annual_rainfall'), rainfall_vis, 'Annual Rainfall')
Map.addLayer(inputLayers.select('wet_months'), wet_months_vis, 'Wet Months')
Map.addLayer(inputLayers.select('rabi_mean_temp'), temp_vis, 'Seasonal Mean Temperature')
Map.addLayer(inputLayers.select('mean_elevation'), elevation_vis, 'Mean Elevation')
Map.centerObject(india, 4)
Map

Map(center=[22.22810924998663, 79.40094700957486], controls=(WidgetControl(options=['position', 'transparent_b…